In [10]:
from importlib.metadata import version
import tiktoken

print("tiktoken version:", version("tiktoken"))

tiktoken version: 0.12.0


In [4]:
tokenizer = tiktoken.get_encoding("gpt2")
text = ("Hello, world! <|endoftext|> This is a test."
        " Let's see how the tokenizer handles special tokens.")

integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
print("Encoded integers:", integers)

Encoded integers: [15496, 11, 995, 0, 220, 50256, 770, 318, 257, 1332, 13, 3914, 338, 766, 703, 262, 11241, 7509, 17105, 2041, 16326, 13]


In [7]:
strings = tokenizer.decode(integers)
print("Decoded string:", strings)

Decoded string: Hello, world! <|endoftext|> This is a test. Let's see how the tokenizer handles special tokens.


In [ ]:
# 未知のトークンを含む場合の動作確認(Akwirw ier)
tokenizer = tiktoken.get_encoding("gpt2")
text = ("Hello, world! <|endoftext|> This is a test. Akwirw ier."
        " Let's see how the tokenizer handles special tokens.")

integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
print("Encoded integers:", integers)

strings = tokenizer.decode(integers)
print("Decoded string:", strings)

Encoded integers: [15496, 11, 995, 0, 220, 50256, 770, 318, 257, 1332, 13, 9084, 86, 343, 86, 220, 959, 13, 3914, 338, 766, 703, 262, 11241, 7509, 17105, 2041, 16326, 13]
Decoded string: Hello, world! <|endoftext|> This is a test. Akwirw ier. Let's see how the tokenizer handles special tokens.


In [10]:
with open("verdict.txt", "r") as f:
    raw_text = f.read()

enc_text = tokenizer.encode(raw_text)
print("Number of tokens in verdict.txt:", len(enc_text))

Number of tokens in verdict.txt: 5145


In [13]:
enc_sample = enc_text[50:]

context_size = 4
x = enc_sample[:context_size]
y = enc_sample[1:context_size + 1]
print("x:", x)
print("y:", y)

for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    print(f"Context: {context}, String: '{tokenizer.decode(context)}' -> Next token: {desired}, String: '{tokenizer.decode([desired])}'")

x: [290, 4920, 2241, 287]
y: [4920, 2241, 287, 257]
Context: [290], String: ' and' -> Next token: 4920, String: ' established'
Context: [290, 4920], String: ' and established' -> Next token: 2241, String: ' himself'
Context: [290, 4920, 2241], String: ' and established himself' -> Next token: 287, String: ' in'
Context: [290, 4920, 2241, 287], String: ' and established himself in' -> Next token: 257, String: ' a'


In [7]:
import torch
from torch.utils.data import Dataset, DataLoader

class GPTDatasetV1(Dataset):
    def __init__(self, text, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []
        token_ids = tokenizer.encode(text)

        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1:i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk, dtype=torch.long))
            self.target_ids.append(torch.tensor(target_chunk, dtype=torch.long))
    
    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

In [8]:
def create_dataloader(text, batch_size=4, max_length=256, stride=128, shuffle=True, drop_last=True, num_workers=0):
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDatasetV1(text, tokenizer, max_length, stride)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last, num_workers=num_workers)
    return dataloader 

In [12]:
with open("verdict.txt", "r") as f:
    raw_text = f.read()

dataloader = create_dataloader(raw_text, batch_size=1, max_length=4, stride=1, shuffle=False, drop_last=False, num_workers=0)

data_iter = iter(dataloader)
first_batch = next(data_iter)
print("First batch", first_batch)

second_batch = next(data_iter)
print("Second batch", second_batch)

First batch [tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]
Second batch [tensor([[ 367, 2885, 1464, 1807]]), tensor([[2885, 1464, 1807, 3619]])]


In [13]:
dataloader = create_dataloader(raw_text, batch_size=8, max_length=4, stride=1, shuffle=False, drop_last=False, num_workers=0)

data_iter = iter(dataloader)
first_batch = next(data_iter)
print("First batch", first_batch)

second_batch = next(data_iter)
print("Second batch", second_batch)

First batch [tensor([[   40,   367,  2885,  1464],
        [  367,  2885,  1464,  1807],
        [ 2885,  1464,  1807,  3619],
        [ 1464,  1807,  3619,   402],
        [ 1807,  3619,   402,   271],
        [ 3619,   402,   271, 10899],
        [  402,   271, 10899,  2138],
        [  271, 10899,  2138,   257]]), tensor([[  367,  2885,  1464,  1807],
        [ 2885,  1464,  1807,  3619],
        [ 1464,  1807,  3619,   402],
        [ 1807,  3619,   402,   271],
        [ 3619,   402,   271, 10899],
        [  402,   271, 10899,  2138],
        [  271, 10899,  2138,   257],
        [10899,  2138,   257,  7026]])]
Second batch [tensor([[10899,  2138,   257,  7026],
        [ 2138,   257,  7026, 15632],
        [  257,  7026, 15632,   438],
        [ 7026, 15632,   438,  2016],
        [15632,   438,  2016,   257],
        [  438,  2016,   257,   922],
        [ 2016,   257,   922,  5891],
        [  257,   922,  5891,  1576]]), tensor([[ 2138,   257,  7026, 15632],
        [  257,  

In [16]:
input_ids = torch.tensor([2,3,5,1])
vocab_size = 6
output_dim = 3

torch.manual_seed(123)
embedding_layer = torch.nn.Embedding(num_embeddings=vocab_size, embedding_dim=output_dim)
print(embedding_layer.weight)
print(embedding_layer(torch.tensor([3])))

Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)
tensor([[-0.4015,  0.9666, -1.1481]], grad_fn=<EmbeddingBackward0>)


In [17]:
vocab_size = 50257  # GPT-2の語彙数
output_dim = 256    # 埋め込みベクトルの次元数
token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

In [19]:
max_length = 4
dataloader = create_dataloader(raw_text, batch_size=8, max_length=max_length, stride=max_length, shuffle=False)

data_iter = iter(dataloader)
input_ids, target_ids = next(data_iter)
print("Input IDs:", input_ids)
print("Target IDs:", target_ids)
print("Input Shape:", input_ids.shape)

Input IDs: tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])
Target IDs: tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])
Input Shape: torch.Size([8, 4])


In [22]:
token_embeddings = token_embedding_layer(input_ids)
print("Text Embedding Shape:", token_embeddings.shape)

Text Embedding Shape: torch.Size([8, 4, 256])


In [21]:
context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)
pos_embeddings = pos_embedding_layer(torch.arange(context_length))
print(pos_embedding_layer)

Embedding(4, 256)


In [27]:
input_embeddings = token_embeddings + pos_embeddings
print(input_embeddings.shape)
print(token_embeddings[0])
print(pos_embeddings[0])
print(input_embeddings[0])

torch.Size([8, 4, 256])
tensor([[ 0.4913,  1.1239,  1.4588,  ..., -0.3995, -1.8735, -0.1445],
        [ 0.4481,  0.2536, -0.2655,  ...,  0.4997, -1.1991, -1.1844],
        [-0.2507, -0.0546,  0.6687,  ...,  0.9618,  2.3737, -0.0528],
        [ 0.9457,  0.8657,  1.6191,  ..., -0.4544, -0.7460,  0.3483]],
       grad_fn=<SelectBackward0>)
tensor([ 1.7375e+00, -5.6195e-01, -6.3027e-01, -4.8483e-01, -1.3660e-01,
         1.7588e+00,  1.8998e+00, -1.4937e-01, -8.3352e-01,  1.0413e+00,
        -7.0012e-01,  2.4318e-01,  1.8356e-01, -5.9014e-01,  3.4127e-02,
         2.1738e+00,  8.1060e-01, -7.8266e-01, -7.7183e-01,  1.2198e+00,
        -4.1258e-01,  1.6117e+00, -4.2123e-01,  6.3503e-01, -6.5394e-01,
         1.8390e+00,  1.3469e+00, -3.2765e-01,  8.7038e-01, -3.2553e-01,
        -1.7075e+00,  9.2959e-01, -6.1158e-01,  6.0666e-01, -1.1067e+00,
         1.0557e+00,  6.3271e-01,  6.5733e-01, -5.3225e-01,  7.9329e-01,
         9.3709e-01, -4.4953e-01,  3.6753e-01, -2.0968e-01,  1.1318e+00,
    